# Gagan Suraksha: TLE Error-Correction Model Training
### Machine Learning Orbital Residual Correction for LEO Satellites

This Google Colab notebook implements the complete data ingestion, physics-informed feature engineering, quantile model training, and horizon evaluation pipeline for the **Gagan Suraksha** Space Situational Awareness (SSA) engine.

----
### Pipeline Overview
1. **Space Weather Ingestion**: Daily solar flux ($F_{10.7}$, 81-day centered average) and geomagnetic planetary indices ($K_p$, $A_p$) from CelesTrak/NOAA SWPC.
2. **Historical TLE Catalog**: Multi-epoch Two-Line Element sets for Low Earth Orbit (LEO, 400–900 km altitude) satellites.
3. **RTN Frame Decomposition**: Analytical SGP4 propagation over 12h to 168h horizons, decomposed into Radial, In-track (along-track), and Cross-track (normal) orbital errors relative to subsequent truth TLEs.
4. **Strict Time-Based Split**: Chronological partitioning (older months $\to$ train, middle $\to$ val, recent $\to$ test) ensuring **zero temporal leakage**.
5. **Quantile XGBoost Models**: $10\%$, $50\%$ (median), and $90\%$ uncertainty quantiles per spatial axis.
6. **Day 1 to 7 Evaluation**: Real computed metric tables and horizon error curves with 10–90% percentile bands.

> **Limitation Notice**: TLE data is sourced from public CelesTrak/Space-Track archives. Reference truth is established using subsequent TLE epoch states propagated with SGP4. High-precision numerical orbit determination (OD) with radar ephemeris is required for operational collision avoidance burns.

## 1. Environment Setup & Dependencies
Install the required libraries for orbital mechanics, space weather ingestion, and gradient boosting.

In [ ]:
!pip install --quiet sgp4 xgboost joblib pandas httpx matplotlib

import os
import sys
import math
import glob
import json
import zipfile
from datetime import datetime, timezone, timedelta
from typing import List, Dict, Tuple, Any, Optional

import httpx
import numpy as np
import pandas as pd
import xgboost as xgb
import joblib
import matplotlib.pyplot as plt
from sgp4.api import Satrec

print(f"XGBoost version: {xgb.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Global Configuration & Paths

In [ ]:
DATA_DIR = "data"
SW_DIR = os.path.join(DATA_DIR, "space_weather")
TLE_DIR = os.path.join(DATA_DIR, "tles")
OUTPUT_DIR = "models"

os.makedirs(SW_DIR, exist_ok=True)
os.makedirs(TLE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

USER_AGENT = "GaganSuraksha-SSA/1.0 (Colab Research; ml-pipeline@gagansuraksha.local)"
CELESTRAK_SW_ALL_URL = "https://celestrak.org/SpaceData/SW-All.csv"
CELESTRAK_SW_5YR_URL = "https://celestrak.org/SpaceData/SW-Last5Years.csv"

FEATURE_NAMES = [
    "horizon_hours",
    "altitude_km",
    "inclination_deg",
    "eccentricity",
    "bstar",
    "mean_motion_rev_day",
    "f107_daily",
    "f107_81d_avg",
    "kp_epoch",
    "ap_epoch",
    "kp_mean_horizon",
    "ap_mean_horizon",
    "object_type_code",
]

TARGET_NAMES = [
    "residual_radial_km",
    "residual_in_track_km",
    "residual_cross_track_km",
]

## 3. Space Weather Data Manager
Downloads and parses solar flux ($F_{10.7}$) and geomagnetic ($K_p$, $A_p$) indices directly from CelesTrak. Fast $O(1)$ dictionary lookup is pre-cached for propagation intervals.

In [ ]:
class SpaceWeatherManager:
    def __init__(self, cache_dir: str):
        self.cache_dir = cache_dir
        self.all_csv_path = os.path.join(cache_dir, "SW-All.csv")
        self.five_yr_csv_path = os.path.join(cache_dir, "SW-Last5Years.csv")
        self.dict_cache: Dict[str, Dict[str, float]] = {}
        self._ensure_and_load_data()

    def _ensure_and_load_data(self) -> None:
        if not os.path.exists(self.all_csv_path) and not os.path.exists(self.five_yr_csv_path):
            self._download_from_celestrak()

        csv_path = self.all_csv_path if os.path.exists(self.all_csv_path) else self.five_yr_csv_path
        print(f"Loading space weather records from {csv_path}...")
        df = pd.read_csv(csv_path, low_memory=False)
        df.columns = [c.strip() for c in df.columns]

        date_col = next((c for c in ["DATE", "Date", "date"] if c in df.columns), None)
        f107_col = next((c for c in ["F10.7_OBS", "F10.7_ADJ", "F10.7"] if c in df.columns), None)
        f107_81d_col = next((c for c in ["F10.7_81_CENTERED", "F10.7_81_AVG", "F10.7_81"] if c in df.columns), None)
        kp_col = next((c for c in ["KP_SUM", "Kp_sum", "KP"] if c in df.columns), None)
        ap_col = next((c for c in ["AP_AVG", "Ap_avg", "AP"] if c in df.columns), None)

        for _, row in df.iterrows():
            d_val = str(row[date_col]).strip()
            if not d_val or d_val.lower() == "nan":
                continue
            try:
                f_daily = float(row[f107_col]) if f107_col and pd.notna(row[f107_col]) else 120.0
                f_81d = float(row[f107_81d_col]) if f107_81d_col and pd.notna(row[f107_81d_col]) else f_daily
                kp_sum = float(row[kp_col]) if kp_col and pd.notna(row[kp_col]) else 20.0
                ap_val = float(row[ap_col]) if ap_col and pd.notna(row[ap_col]) else 10.0
                self.dict_cache[d_val] = {
                    "f107_daily": f_daily,
                    "f107_81d": f_81d,
                    "kp": kp_sum / 8.0,
                    "ap": ap_val,
                }
            except Exception:
                continue

        print(f"Loaded {len(self.dict_cache)} daily space-weather records.")

    def _download_from_celestrak(self) -> None:
        for url, path in [(CELESTRAK_SW_ALL_URL, self.all_csv_path), (CELESTRAK_SW_5YR_URL, self.five_yr_csv_path)]:
            try:
                print(f"Downloading space weather indices from {url}...")
                with httpx.Client(timeout=30.0, headers={"User-Agent": USER_AGENT}) as client:
                    resp = client.get(url)
                    if resp.status_code == 200 and len(resp.text) > 1000:
                        with open(path, "w", encoding="utf-8") as f:
                            f.write(resp.text)
                        print(f"Saved {path} ({len(resp.text):,} bytes).")
                        return
            except Exception as e:
                print(f"Failed downloading {url}: {e}")
        raise RuntimeError("Cannot download space weather from CelesTrak. No synthetic fallback.")

    def get_indices_at_date(self, date_str: str) -> Dict[str, float]:
        return self.dict_cache.get(date_str, {"f107_daily": 120.0, "f107_81d": 120.0, "kp": 2.5, "ap": 10.0})

    def get_horizon_indices(self, start_dt: datetime, end_dt: datetime) -> Tuple[float, float]:
        cur = start_dt.date()
        stop = end_dt.date()
        kp_vals, ap_vals = [], []
        while cur <= stop:
            d_str = cur.strftime("%Y-%m-%d")
            entry = self.dict_cache.get(d_str)
            if entry:
                kp_vals.append(entry["kp"])
                ap_vals.append(entry["ap"])
            cur += timedelta(days=1)
        return (float(np.mean(kp_vals)) if kp_vals else 2.5, float(np.mean(ap_vals)) if ap_vals else 10.0)

sw_manager = SpaceWeatherManager(SW_DIR)

## 4. Ingestion of Historical LEO TLE Archives
Downloads public historical TLE archives from CelesTrak for Indian Earth Observation satellites (IRS-1A, IRS-1B, IRS-1C, IRS-1D, IRS-P2, IRS-P3, IRS-P4, IRS-P6 / ResourceSat) and international benchmark LEO missions (SPOT, TERRA, AQUA, TRMM, ENVISAT, METEOR).

In [ ]:
ARCHIVE_FILES = [
    "1988-021.zip",  # IRS-1A (18960)
    "1991-061.zip",  # IRS-1B (21688)
    "1994-063.zip",  # IRS-P2 (23323)
    "1995-072.zip",  # IRS-1C (23751)
    "1996-017.zip",  # IRS-P3 (23827)
    "1997-057.zip",  # IRS-1D (24971)
    "1999-029.zip",  # IRS-P4 Oceansat (25758)
    "2003-046.zip",  # IRS-P6 ResourceSat-1 (28051)
    "1986-019.zip",  # SPOT 1 (16613)
    "1990-005.zip",  # SPOT 2 (20436)
    "1993-061.zip",  # SPOT 3 (22823)
    "1998-017.zip",  # SPOT 4 (25260)
    "2002-021.zip",  # SPOT 5 (27421)
    "1999-068.zip",  # TERRA (25994)
    "2002-022.zip",  # AQUA (27424)
    "1997-074.zip",  # TRMM (25063)
    "2002-009.zip",  # ENVISAT (27386)
]

for arc in ARCHIVE_FILES:
    url = f"https://celestrak.org/NORAD/archives/legacy/{arc}"
    local_zip = os.path.join(TLE_DIR, arc)
    if not os.path.exists(local_zip):
        try:
            with httpx.Client(timeout=30.0, headers={"User-Agent": USER_AGENT}) as client:
                r = client.get(url)
                if r.status_code == 200:
                    with open(local_zip, "wb") as f:
                        f.write(r.content)
                    with zipfile.ZipFile(local_zip, "r") as z:
                        z.extractall(TLE_DIR)
                    print(f"Extracted {arc} ({len(r.content):,} bytes)")
        except Exception as e:
            print(f"Warning downloading {arc}: {e}")

tle_files = glob.glob(os.path.join(TLE_DIR, "*.txt")) + glob.glob(os.path.join(TLE_DIR, "*.tle"))
print(f"Found {len(tle_files)} raw TLE files ready for processing.")

## 5. TLE Validation & Orbital Element Parsing
Parses Two-Line Element records with SGP4 checksum and Kozai mean motion verification.

In [ ]:
class TLERecord:
    def __init__(self, name: str, line1: str, line2: str, object_type: str = "UNKNOWN"):
        self.name = name.strip()
        self.line1 = line1.strip()
        self.line2 = line2.strip()
        self.object_type = object_type

        self.satrec = Satrec.twoline2rv(self.line1, self.line2)
        if self.satrec.error != 0:
            raise ValueError(f"SGP4 error {self.satrec.error}")

        self.norad_id = int(self.satrec.satnum)
        self.jd_epoch = self.satrec.jdsatepoch + self.satrec.jdsatepochF
        self.epoch_ts = (self.jd_epoch - 2440587.5) * 86400.0
        self.epoch_dt = datetime(1970, 1, 1, tzinfo=timezone.utc) + timedelta(seconds=self.epoch_ts)

        self.mean_motion_rev_day = float(self.satrec.no_kozai * 1440.0 / (2.0 * math.pi))
        n_rad_s = self.satrec.no_kozai / 60.0
        if n_rad_s > 0:
            self.semi_major_axis_km = float((398600.4418 / (n_rad_s ** 2)) ** (1.0 / 3.0))
            self.altitude_km = float(self.semi_major_axis_km - 6378.137)
        else:
            self.semi_major_axis_km = 7000.0
            self.altitude_km = 620.0

        self.inclination_deg = float(self.satrec.inclo * 180.0 / math.pi)
        self.eccentricity = float(self.satrec.ecco)
        self.bstar = float(self.satrec.bstar)

def parse_tle_text(text: str) -> List[TLERecord]:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    records: List[TLERecord] = []
    i = 0
    while i < len(lines):
        if lines[i].startswith("1 ") and i + 1 < len(lines) and lines[i + 1].startswith("2 "):
            name = f"SAT_{lines[i][2:7].strip()}"
            l1, l2 = lines[i], lines[i + 1]
            i += 2
        elif i + 2 < len(lines) and lines[i + 1].startswith("1 ") and lines[i + 2].startswith("2 "):
            name, l1, l2 = lines[i], lines[i + 1], lines[i + 2]
            i += 3
        else:
            i += 1
            continue
        try:
            records.append(TLERecord(name, l1, l2))
        except Exception:
            pass
    return records

all_records: List[TLERecord] = []
for fpath in tle_files:
    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
        all_records.extend(parse_tle_text(f.read()))

print(f"Parsed {len(all_records):,} valid historical TLE records across LEO satellites.")

## 6. RTN Orbit Frame Decomposition & Pair Generation
For each chronological TLE pair $(A, B)$ where $t_B > t_A$ ($12\text{h} \le \Delta t \le 168\text{h}$):
1. Propagate TLE $A$ with SGP4 to epoch $t_B$.
2. Decompose error vector $\Delta \vec{r} = \vec{r}_{prop} - \vec{r}_{true}$ into the orbital RTN frame:
   - **Radial (R)**: $\hat{r} = \frac{\vec{r}}{\|\vec{r}\|}$
   - **Cross-track / Normal (N)**: $\hat{n} = \frac{\vec{r} \times \vec{v}}{\|\vec{r} \times \vec{v}\|}$
   - **In-track / Along-track (T)**: $\hat{t} = \hat{n} \times \hat{r}$

In [ ]:
def decompose_rtn(r1: np.ndarray, v1: np.ndarray, r2: np.ndarray) -> Tuple[float, float, float]:
    r1_norm = np.linalg.norm(r1)
    r_hat = r1 / r1_norm if r1_norm > 1e-6 else np.array([1.0, 0.0, 0.0])
    h = np.cross(r1, v1)
    h_norm = np.linalg.norm(h)
    n_hat = h / h_norm if h_norm > 1e-6 else np.array([0.0, 0.0, 1.0])
    t_hat = np.cross(n_hat, r_hat)
    t_hat = t_hat / np.linalg.norm(t_hat)

    delta_r = r1 - r2
    res_r = float(np.dot(delta_r, r_hat))
    res_t = float(np.dot(delta_r, t_hat))
    res_n = float(np.dot(delta_r, n_hat))
    return res_r, res_t, res_n

sat_map: Dict[int, List[TLERecord]] = {}
for r in all_records:
    if 400.0 <= r.altitude_km <= 900.0:
        sat_map.setdefault(r.norad_id, []).append(r)

rows = []
for norad_id, tles in sat_map.items():
    tles.sort(key=lambda x: x.epoch_ts)
    n_tles = len(tles)
    if n_tles < 2:
        continue
    stride = max(1, (n_tles - 2) // 150)
    pairs_count = 0
    for i in range(0, n_tles - 1, stride):
        tle_a = tles[i]
        for j in range(i + 1, min(i + 20, n_tles)):
            tle_b = tles[j]
            horizon_hours = (tle_b.epoch_ts - tle_a.epoch_ts) / 3600.0
            if horizon_hours < 12.0:
                continue
            if horizon_hours > 168.0:
                break

            jd_target = tle_b.jd_epoch
            jd_int = int(jd_target)
            fr = jd_target - jd_int
            err_a, r_a, v_a = tle_a.satrec.sgp4(jd_int, fr)
            err_b, r_b, v_b = tle_b.satrec.sgp4(jd_int, fr)
            if err_a != 0 or err_b != 0:
                continue

            r_a_vec, v_a_vec, r_b_vec = np.array(r_a), np.array(v_a), np.array(r_b)
            res_r, res_t, res_n = decompose_rtn(r_a_vec, v_a_vec, r_b_vec)
            base_3d = float(np.linalg.norm(r_a_vec - r_b_vec))

            date_str = tle_a.epoch_dt.strftime("%Y-%m-%d")
            sw = sw_manager.get_indices_at_date(date_str)
            m_kp, m_ap = sw_manager.get_horizon_indices(tle_a.epoch_dt, tle_b.epoch_dt)

            rows.append({
                "norad_id": norad_id,
                "satellite_name": tle_a.name,
                "epoch_a_ts": tle_a.epoch_ts,
                "epoch_b_ts": tle_b.epoch_ts,
                "horizon_hours": horizon_hours,
                "altitude_km": tle_a.altitude_km,
                "inclination_deg": tle_a.inclination_deg,
                "eccentricity": tle_a.eccentricity,
                "bstar": tle_a.bstar,
                "mean_motion_rev_day": tle_a.mean_motion_rev_day,
                "f107_daily": sw["f107_daily"],
                "f107_81d_avg": sw["f107_81d"],
                "kp_epoch": sw["kp"],
                "ap_epoch": sw["ap"],
                "kp_mean_horizon": m_kp,
                "ap_mean_horizon": m_ap,
                "object_type_code": 1,
                "residual_radial_km": res_r,
                "residual_in_track_km": res_t,
                "residual_cross_track_km": res_n,
                "baseline_3d_error_km": base_3d,
            })
            pairs_count += 1
            if pairs_count >= 150:
                break
        if pairs_count >= 150:
            break

df_pairs = pd.DataFrame(rows)
print(f"Generated {len(df_pairs):,} residual observation pairs across {df_pairs['norad_id'].nunique()} LEO satellites.")

## 7. Strict Time-Based Split (Zero Leakage)
To prevent data leakage, each satellite's timeline is partitioned chronologically:
- **Train (70%)**: Historical early lifespan pairs.
- **Validation (15%)**: Middle lifespan pairs.
- **Test (15%)**: Strictly future / most recent period pairs.

No test sample contains data from future epochs leaking backward into training.

In [ ]:
train_dfs, val_dfs, test_dfs = [], [], []
for norad_id, group in df_pairs.groupby("norad_id"):
    g_sorted = group.sort_values(by="epoch_b_ts").reset_index(drop=True)
    n = len(g_sorted)
    if n < 5:
        train_dfs.append(g_sorted)
        continue
    n_tr = int(n * 0.70)
    n_v = int(n * 0.15)
    train_dfs.append(g_sorted.iloc[:n_tr])
    val_dfs.append(g_sorted.iloc[n_tr:n_tr + n_v])
    test_dfs.append(g_sorted.iloc[n_tr + n_v:])

df_train = pd.concat(train_dfs, ignore_index=True)
df_val = pd.concat(val_dfs, ignore_index=True)
df_test = pd.concat(test_dfs, ignore_index=True)

print(f"Time-based split: Train={len(df_train)} pairs, Val={len(df_val)} pairs, Test={len(df_test)} pairs")

## 8. Multi-Target Quantile XGBoost Training
Trains 9 gradient boosted quantile trees (quantiles: $0.10, 0.50, 0.90$) across Radial, In-track, and Cross-track spatial coordinates.

In [ ]:
class TLEErrorCorrectionModel:
    def __init__(self):
        self.models = {
            "residual_radial_km": {},
            "residual_in_track_km": {},
            "residual_cross_track_km": {},
        }
        self.quantiles = [0.10, 0.50, 0.90]

    def fit(self, df_tr: pd.DataFrame, df_v: pd.DataFrame):
        X_tr = df_tr[FEATURE_NAMES].values
        X_v = df_v[FEATURE_NAMES].values

        for target in TARGET_NAMES:
            y_tr = df_tr[target].values
            y_v = df_v[target].values
            dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=FEATURE_NAMES)
            dv = xgb.DMatrix(X_v, label=y_v, feature_names=FEATURE_NAMES)

            print(f"Training quantiles for target: {target}...")
            for q in self.quantiles:
                params = {
                    "objective": "reg:quantileerror",
                    "quantile_alpha": q,
                    "max_depth": 5,
                    "learning_rate": 0.05,
                    "subsample": 0.85,
                    "colsample_bytree": 0.85,
                    "tree_method": "hist",
                }
                bst = xgb.train(params, dtr, num_boost_round=150, evals=[(dtr, "train"), (dv, "val")], verbose_eval=False)
                self.models[target][q] = bst

    def predict(self, X: np.ndarray) -> Dict[str, Dict[float, np.ndarray]]:
        dm = xgb.DMatrix(X, feature_names=FEATURE_NAMES)
        return {
            target: {q: self.models[target][q].predict(dm) for q in self.quantiles}
            for target in TARGET_NAMES
        }

model = TLEErrorCorrectionModel()
model.fit(df_train, df_val)
print("Model training complete!")

## 9. Evaluation Across Horizons (Day 1 to 7)
Computes real median and 90th percentile 3D position errors for baseline SGP4 vs ML-corrected predictions.

In [ ]:
X_test = df_test[FEATURE_NAMES].values
preds = model.predict(X_test)

p_r = preds["residual_radial_km"][0.50]
p_t = preds["residual_in_track_km"][0.50]
p_n = preds["residual_cross_track_km"][0.50]

t_r = df_test["residual_radial_km"].values
t_t = df_test["residual_in_track_km"].values
t_n = df_test["residual_cross_track_km"].values

base_3d = np.sqrt(t_r**2 + t_t**2 + t_n**2)
corr_3d = np.sqrt((t_r - p_r)**2 + (t_t - p_t)**2 + (t_n - p_n)**2)

df_eval = df_test.copy()
df_eval["base_3d"] = base_3d
df_eval["corr_3d"] = corr_3d

horizon_bins = [
    ("Day 1", 12.0, 36.0),
    ("Day 2", 36.0, 60.0),
    ("Day 3", 60.0, 84.0),
    ("Day 4", 84.0, 108.0),
    ("Day 5", 108.0, 132.0),
    ("Day 6", 132.0, 156.0),
    ("Day 7", 156.0, 180.0),
]

table_rows = []
for label, h_min, h_max in horizon_bins:
    sub = df_eval[(df_eval["horizon_hours"] >= h_min) & (df_eval["horizon_hours"] < h_max)]
    if len(sub) > 0:
        b_med = float(np.median(sub["base_3d"]))
        b_p90 = float(np.percentile(sub["base_3d"], 90))
        c_med = float(np.median(sub["corr_3d"]))
        c_p90 = float(np.percentile(sub["corr_3d"], 90))
        imp = ((b_med - c_med) / b_med) * 100.0
    else:
        b_med, b_p90, c_med, c_p90, imp = 0, 0, 0, 0, 0
    table_rows.append({
        "Horizon": label,
        "Samples": len(sub),
        "Baseline Med (km)": round(b_med, 2),
        "Baseline p90 (km)": round(b_p90, 2),
        "Corrected Med (km)": round(c_med, 2),
        "Corrected p90 (km)": round(c_p90, 2),
        "Improvement (%)": round(imp, 1),
    })

eval_table = pd.DataFrame(table_rows)
display(eval_table)

day7_imp = eval_table.loc[eval_table["Horizon"] == "Day 7", "Improvement (%)"].values[0]
print(f"\nComputed Caption: Residual correction using space-weather features reduced median position error by {day7_imp}% at Day 7.")

## 10. Horizon Error Curve Visualization
Plots the Orbit Error Horizon curve comparing baseline SGP4 vs ML-corrected positions with 10–90 percentile uncertainty bands.

In [ ]:
days = [row["Horizon"] for row in table_rows]
b_meds = [row["Baseline Med (km)"] for row in table_rows]
c_meds = [row["Corrected Med (km)"] for row in table_rows]
b_p90s = [row["Baseline p90 (km)"] for row in table_rows]
c_p90s = [row["Corrected p90 (km)"] for row in table_rows]

plt.figure(figsize=(10, 6))
plt.plot(days, b_meds, 'o-', color='#ef4444', label='Baseline SGP4 (Median)')
plt.plot(days, c_meds, 's-', color='#06b6d4', label='ML-Corrected (Median)')
plt.fill_between(days, b_meds, b_p90s, color='#ef4444', alpha=0.15, label='Baseline 90th percentile band')
plt.fill_between(days, c_meds, c_p90s, color='#06b6d4', alpha=0.20, label='ML-Corrected 90th percentile band')

plt.yscale('log')
plt.xlabel('Propagation Horizon', fontsize=12)
plt.ylabel('3D Position Error (km, log scale)', fontsize=12)
plt.title('Orbit Error Horizon: Baseline SGP4 vs ML-Corrected (Gagan Suraksha)', fontsize=14, fontweight='bold')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 11. Artifact Export
Saves the model dictionary and `metadata.json` for deployment.

In [ ]:
model_export_path = os.path.join(OUTPUT_DIR, "tle_error_xgb_v1.0.0.joblib")
joblib.dump({
    "model_version": "v1.0.0",
    "feature_names": FEATURE_NAMES,
    "target_names": TARGET_NAMES,
    "quantiles": model.quantiles,
    "models": model.models,
}, model_export_path)

metadata_dict = {
    "model_version": "v1.0.0",
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "reference_truth": "Later TLE epoch state (SGP4 propagation at epoch tB)",
    "data_source": "CelesTrak historical TLE archives & SpaceData (SW-All.csv, SW-Last5Years.csv)",
    "evaluation_table": table_rows,
}

with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
    json.dump(metadata_dict, f, indent=2)

print(f"Saved {model_export_path} and metadata.json.")